In [74]:
from pathlib import Path
import sys
import pandas as pd
import vitaldb

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.preprocessing import process_case, TRACK_NAMES

print("Project root:", project_root)
print("50-patient pipeline ready!")

Project root: c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction
50-patient pipeline ready!


In [75]:
caseids = vitaldb.find_cases(TRACK_NAMES)

print("Total eligible cases:", len(caseids))
print("First 10 case IDs:", caseids[:10])

C:\Users\samat\AppData\Local\Temp\ipykernel_10580\2083261614.py:1: DeprecationWarning: find_cases() relies on the per-track 'trks' index, which is deprecated and may be removed in a future release.
  caseids = vitaldb.find_cases(TRACK_NAMES)


Total eligible cases: 3730
First 10 case IDs: [1, 3, 4, 7, 10, 12, 13, 14, 16, 17]


In [76]:
selected_caseids = caseids[:50]

print("Number of selected patients:", len(selected_caseids))
print(selected_caseids)

Number of selected patients: 50
[1, 3, 4, 7, 10, 12, 13, 14, 16, 17, 19, 20, 22, 24, 25, 26, 27, 28, 29, 31, 32, 34, 38, 41, 43, 44, 46, 49, 50, 51, 52, 53, 55, 56, 58, 59, 60, 61, 64, 65, 66, 67, 68, 69, 70, 74, 75, 76, 77, 79]


In [77]:
processed_cases = []
failed_cases = []

for number, caseid in enumerate(selected_caseids, start=1):

    print(
        f"[{number}/{len(selected_caseids)}] "
        f"Processing case {caseid}..."
    )

    try:
        df_case = process_case(caseid)

        processed_cases.append(df_case)

        print(
            f"Completed case {caseid}: "
            f"{len(df_case)} rows"
        )

    except Exception as e:

        failed_cases.append({
            "caseid": caseid,
            "error": str(e)
        })

        print(
            f"FAILED case {caseid}: {e}"
        )

[1/50] Processing case 1...
Completed case 1: 193 rows
[2/50] Processing case 3...
Completed case 3: 74 rows
[3/50] Processing case 4...
Completed case 4: 350 rows
[4/50] Processing case 7...
Completed case 7: 263 rows
[5/50] Processing case 10...
Completed case 10: 350 rows
[6/50] Processing case 12...
Completed case 12: 521 rows
[7/50] Processing case 13...
Completed case 13: 181 rows
[8/50] Processing case 14...
Completed case 14: 69 rows
[9/50] Processing case 16...
Completed case 16: 215 rows
[10/50] Processing case 17...
Completed case 17: 340 rows
[11/50] Processing case 19...
Completed case 19: 460 rows
[12/50] Processing case 20...
Completed case 20: 442 rows
[13/50] Processing case 22...
Completed case 22: 240 rows
[14/50] Processing case 24...
Completed case 24: 109 rows
[15/50] Processing case 25...
Completed case 25: 248 rows
[16/50] Processing case 26...
Completed case 26: 179 rows
[17/50] Processing case 27...
Completed case 27: 298 rows
[18/50] Processing case 28...
Com

In [78]:
print("Successful patients:", len(processed_cases))
print("Failed patients:", len(failed_cases))

Successful patients: 50
Failed patients: 0


In [79]:
failed_cases[:10]

[]

In [80]:
df_50 = pd.concat(
    processed_cases,
    ignore_index=True
)

print("Combined dataset shape:", df_50.shape)
print("Unique patients:", df_50["caseid"].nunique())

Combined dataset shape: (12515, 23)
Unique patients: 50


In [81]:
print(
    "Duplicate patient-minute rows:",
    df_50.duplicated(
        subset=["caseid", "minute"]
    ).sum()
)

Duplicate patient-minute rows: 0


In [82]:
print(
    "Rows with future target:",
    df_50["future_spo2_5min"].notna().sum()
)

print(
    "Rows without future target:",
    df_50["future_spo2_5min"].isna().sum()
)

Rows with future target: 11830
Rows without future target: 685


In [83]:
future_columns = [
    col
    for col in df_50.columns
    if "future" in col.lower()
]

future_columns

['future_spo2_5min']

In [84]:
output_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50.csv"
)

df_50.to_csv(
    output_file,
    index=False
)

print("50-patient dataset saved to:")
print(output_file)

50-patient dataset saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\multi_patient_50.csv


In [85]:
failed_file = (
    project_root
    / "data"
    / "processed"
    / "failed_cases_50.csv"
)

pd.DataFrame(
    failed_cases
).to_csv(
    failed_file,
    index=False
)

print("Failure log saved to:")
print(failed_file)

Failure log saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\failed_cases_50.csv


In [ ]:
print("=== FINAL DATASET CHECK ===")
print()

print("Successful patients:", len(processed_cases))
print("Failed patients:", len(failed_cases))
print()

print("Dataset shape:", df_50.shape)
print(
    "Unique patients:",
    df_50["caseid"].nunique()
)
print(
    "Duplicate patient-minute rows:",
    df_50.duplicated(
        subset=["caseid", "minute"]
    ).sum()
)
print()

print(
    "Rows with target:",
    df_50["future_spo2_5min"].notna().sum()
)
print(
    "Rows without target:",
    df_50["future_spo2_5min"].isna().sum()
)
print()

print("Future columns:", future_columns)

=== FINAL DATASET CHECK ===

Successful patients: 50
Failed patients: 0

Dataset shape: (12515, 23)
Unique patients: 50
Duplicate patient-minute rows: 0

Rows with target: 11830
Rows without target: 685

Future columns: ['future_spo2_5min']


: 